In [ ]:
import os
import torch
import random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
from predict import HybridPredictor
import cv2

ref_images = ['ref1.jpg', 'ref2.jpg', 'ref3.jpg']
predictor = HybridPredictor(
    reference_images=ref_images,
    config_path='config/config.yaml'
)

In [ ]:
from pathlib import Path

test_cases = []
dataset_path = Path('test_dataset/samples')

for video_dir in sorted(dataset_path.iterdir()):
    if video_dir.is_dir():
        video_file = video_dir / 'drone_video.mp4'
        object_images_dir = video_dir / 'object_images'
        if video_file.exists() and object_images_dir.exists():
            ref_imgs = sorted(list(object_images_dir.glob('*.jpg')) + list(object_images_dir.glob('*.png')))[:3]
            test_cases.append({
                'video_id': video_dir.name,
                'video_path': str(video_file),
                'ref_images': [str(img) for img in ref_imgs]
            })

In [ ]:
from time import time
import json
import csv

all_predicted_time = []
all_result = []

for test_case in test_cases:
    video_id = test_case['video_id']
    video_path = test_case['video_path']
    ref_images = test_case['ref_images']
    
    t1 = time()
    
    predictor_instance = HybridPredictor(
        reference_images=ref_images,
        config_path='config/config.yaml'
    )
    
    predictions = predictor_instance.process_video(
        video_path=video_path,
        output_path=None,
        visualize=False
    )
    
    t2 = time()
    
    predicted_time = int(t2*1000 - t1*1000)
    all_predicted_time.append((video_id, predicted_time))
    
    result = {
        'video_id': video_id,
        'detections': [{'bboxes': predictions}] if len(predictions) > 0 else []
    }
    all_result.append(result)

with open('submission.json', 'w') as f:
    json.dump(all_result, f, indent=2)

with open('time_submission.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['video_id', 'time_ms'])
    writer.writerows(all_predicted_time)